# Narada — GRPO Training Notebook

**Meta × PyTorch OpenEnv Hackathon × Scaler School of Technology**

Trains Qwen3-1.7B as the Detective agent on the Narada rare-disease diagnosis environment using GRPO (Group Relative Policy Optimisation).

## Setup requirements
- Colab: A100 or L4 (≥16 GB VRAM). T4 works but is slow.
- Runtime: GPU, High-RAM

## What this notebook does
1. Loads `Qwen/Qwen3-1.7B` with Unsloth 4-bit quantisation
2. Connects to the live Narada environment at `https://krishvenky-narada-env.hf.space`
3. Runs GRPO training in curriculum order: `monogenic → oligogenic → phenotype_mismatch`
4. Evaluates on held-out seeds after each phase
5. Pushes the trained LoRA adapter to your HF Hub

In [ ]:
# -- Cell 1: Install dependencies --
# Run once. Restart runtime after this cell.

# unsloth_zoo must be installed first
!pip install -q unsloth_zoo

!pip install -q \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.9.0" \
    "peft>=0.10.0" \
    "accelerate>=0.30.0" \
    "bitsandbytes>=0.43.0" \
    "websockets>=12.0" \
    "pydantic>=2.7.0" \
    "openai>=1.30.0" \
    "nest_asyncio>=1.6.0" \
    "matplotlib>=3.7.0"

print("Install complete -- restart runtime now if this is the first run.")

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────

import os

# ---------- Required: set your HF token ----------
HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste your HF token (write access): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN

# ---------- Model ----------
BASE_MODEL      = "Qwen/Qwen3-1.7B"          # base model for training
ADAPTER_NAME    = "narada-detective-lora"     # local adapter save name
HF_PUSH_REPO    = ""                          # optional: "YourUser/narada-detective"

# ---------- Environment ----------
ENV_URL         = "https://krishvenky-narada-env.hf.space"

# ---------- Training ----------
LORA_RANK       = 16
MAX_SEQ_LEN     = 2048
GRPO_BATCH_SIZE = 4     # number of completions per prompt (G)
MINI_BATCH_SIZE = 2
GRAD_ACCUM      = 4
LR              = 5e-6
WARMUP_STEPS    = 20

CURRICULUM = [
    {"task": "monogenic",          "steps": 80,  "max_ep_steps": 15},
    {"task": "oligogenic",         "steps": 60,  "max_ep_steps": 25},
    {"task": "phenotype_mismatch", "steps": 60,  "max_ep_steps": 20},
]

EVAL_SEEDS = [42, 7, 999, 1337, 2024]  # held-out seeds for evaluation

print(f"Config ready. Base model: {BASE_MODEL}")

In [ ]:
# -- Cell 3: Load model with Unsloth --
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

import torch

# torchao 0.17+ calls torch.utils._pytree.register_constant (PyTorch 2.7+).
# Stub it for PyTorch 2.6 -- we use bitsandbytes, not torchao.
if not hasattr(torch.utils._pytree, "register_constant"):
    torch.utils._pytree.register_constant = lambda cls: cls

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    token           = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha      = LORA_RANK * 2,
    lora_dropout    = 0.0,
    bias            = "none",
    use_gradient_checkpointing = "unsloth",
    random_state    = 42,
)

print(f"Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

_orig_apply = tokenizer.apply_chat_template
def _apply_no_think(*args, **kwargs):
    kwargs["enable_thinking"] = False
    return _orig_apply(*args, **kwargs)
tokenizer.apply_chat_template = _apply_no_think
print("Thinking mode disabled.")

In [ ]:
# ── Cell 4: Environment client + helpers ──────────────────────────────────────

import asyncio, json, re, textwrap
import nest_asyncio
nest_asyncio.apply()
from typing import List, Dict, Any, Optional
import websockets

SYSTEM_PROMPT = textwrap.dedent("""
You are an expert clinical geneticist. Generate a DIAGNOSTIC PLAN: 3-5 JSON action blocks
to navigate the gene-disease knowledge graph and identify the causal variant.

Output each action block on its own line. End with flag_causal when you are confident.

ACTIONS (one JSON per line, no other text):
  {"action_type": "hop",          "node_id": "<id>",       "reasoning": "<one sentence>"}
  {"action_type": "flag_causal",  "variant_id": "VAR:...", "reasoning": "<one sentence>"}
  {"action_type": "backtrack",                              "reasoning": "<one sentence>"}
  {"action_type": "summarise_trail",                        "reasoning": "<one sentence>"}

STRATEGY:
  1. Navigate phenotype -> disease -> gene -> variant chains.
  2. BRCA1/TP53 is a DECOY if phenotypes are cardiac/neurological -- skip it.
  3. Oligogenic: flag ALL causal variants, not just the first one.
  4. Flag before step 8 for a timing bonus.
  5. ABSENT PHENOTYPES are strong rule-out signals -- use them.
""").strip()


def format_obs(obs: Dict[str, Any]) -> str:
    lines = [
        f"STEP {obs['step']}/{obs['max_steps']} | Task: {obs['task_type']}",
        "",
        "PATIENT PHENOTYPES (present):",
    ]
    for hid, name in zip(obs["patient_phenotypes"], obs["phenotype_names"]):
        lines.append(f"  + {hid} -- {name}")

    absent_ids = obs.get("phenotypes_absent") or []
    absent_names = obs.get("phenotype_absent_names") or []
    if absent_ids:
        lines += ["", "ABSENT PHENOTYPES (rule-out signal):"]
        for hid, name in zip(absent_ids, absent_names):
            lines.append(f"  - {hid} -- {name}")

    n = obs["current_node"]
    lines += [
        "",
        f"CURRENT NODE: [{n['type'].upper()}] {n['name']} ({n['id']})",
        f"  Neighbors ({len(n['connected_node_ids'])}): {', '.join(n['connected_node_ids'][:8])}",
    ]

    if obs.get("trail"):
        trail_names = [f"{t['name']}({t['id']})" for t in obs["trail"][-4:]]
        lines.append(f"  Trail: {' -> '.join(trail_names)}")

    lines += ["", "CANDIDATE VARIANTS:"]
    for v in obs["candidate_variants"]:
        lines.append(
            f"  {v['id']} | {v['gene']} | {v['variant_type']} "
            f"| path={v['pathogenicity_score']:.2f} | {v['clinical_significance']}"
        )

    lines.append(f"\nStep reward: {obs['step_reward']:+.4f} | Cumulative: {obs['cumulative_reward']:.4f}")
    lines.append("Generate your diagnostic plan (3-5 JSON action blocks):")
    return "\n".join(lines)


def parse_all_actions(text: str) -> List[Dict[str, Any]]:
    """Extract all JSON action blocks from a multi-step completion."""
    actions = []
    for m in re.finditer(r'\{[^{}]*"action_type"[^{}]*\}', text, re.DOTALL):
        try:
            d = json.loads(m.group(0))
            atype = str(d.get("action_type", "")).lower()
            if atype not in ("hop", "flag_causal", "backtrack", "summarise_trail", "request_lab"):
                continue
            actions.append({
                "action_type": atype,
                "node_id": str(d["node_id"]) if d.get("node_id") else None,
                "variant_id": str(d["variant_id"]) if d.get("variant_id") else None,
                "reasoning": str(d.get("reasoning", ""))[:200],
            })
            if atype == "flag_causal":
                break  # stop at first flag
        except Exception:
            continue
    return actions or [{"action_type": "summarise_trail", "reasoning": "fallback"}]


def parse_action(text: str) -> Dict[str, Any]:
    """Single-action parser — used in eval/inference loops."""
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return {"action_type": "summarise_trail", "reasoning": "fallback"}
    try:
        d = json.loads(m.group(0))
        atype = str(d.get("action_type", "summarise_trail")).lower()
        if atype not in ("hop", "flag_causal", "backtrack", "summarise_trail", "request_lab"):
            atype = "summarise_trail"
        return {
            "action_type": atype,
            "node_id": str(d["node_id"]) if d.get("node_id") else None,
            "variant_id": str(d["variant_id"]) if d.get("variant_id") else None,
            "reasoning": str(d.get("reasoning", ""))[:200],
        }
    except Exception:
        return {"action_type": "summarise_trail", "reasoning": "parse error"}


async def run_episode_async(
    task_type: str,
    actions: List[Dict[str, Any]],
    seed: Optional[int] = None,
) -> float:
    ws_url = ENV_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"
    async with websockets.connect(ws_url, open_timeout=30, ping_interval=20) as ws:
        reset_msg = {"type": "reset", "task_type": task_type}
        if seed is not None:
            reset_msg["seed"] = seed
        await ws.send(json.dumps(reset_msg))
        raw = json.loads(await ws.recv())
        if raw.get("type") == "error":
            return 0.1

        obs = raw["data"]["observation"]
        last_reward = 0.1

        for action in actions:
            if obs.get("done"):
                break
            await ws.send(json.dumps({"type": "step", "action": action}))
            raw = json.loads(await ws.recv())
            if raw.get("type") == "error":
                break
            data = raw["data"]
            obs = data["observation"]
            last_reward = data["reward"]
            if obs.get("done"):
                return float(last_reward)

        return float(last_reward)


async def collect_episode_async(task_type: str, seed: Optional[int] = None) -> List[Dict]:
    ws_url = ENV_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"
    steps = []
    async with websockets.connect(ws_url, open_timeout=30, ping_interval=20) as ws:
        reset_msg = {"type": "reset", "task_type": task_type}
        if seed is not None:
            reset_msg["seed"] = seed
        await ws.send(json.dumps(reset_msg))
        raw = json.loads(await ws.recv())
        if raw.get("type") == "error":
            return steps
        obs = raw["data"]["observation"]
        steps.append({"prompt": format_obs(obs), "obs": obs, "task_type": task_type, "seed": seed})
    return steps


def run_episode(task_type, actions, seed=None):
    return asyncio.get_event_loop().run_until_complete(
        run_episode_async(task_type, actions, seed)
    )


def collect_episode(task_type, seed=None):
    return asyncio.get_event_loop().run_until_complete(
        collect_episode_async(task_type, seed)
    )

In [ ]:
# -- Cell 5: Build prompt dataset (parallel) --

from datasets import Dataset
import random

N_SEEDS_PER_TASK = 40
random.seed(42)
train_seeds = random.sample(range(1, 10000), N_SEEDS_PER_TASK * 3)

async def _collect_one(task, seed):
    steps = await collect_episode_async(task, seed=seed)
    if not steps:
        return None
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": steps[0]["prompt"]},
        ],
        "task_type": task,
        "seed": seed,
    }

async def _collect_all():
    pairs = []
    for i, phase in enumerate(CURRICULUM):
        task = phase["task"]
        for seed in train_seeds[i * N_SEEDS_PER_TASK : (i + 1) * N_SEEDS_PER_TASK]:
            pairs.append((task, seed))
    results = await asyncio.gather(*[_collect_one(t, s) for t, s in pairs])
    return [r for r in results if r is not None]

print(f"Collecting {N_SEEDS_PER_TASK * len(CURRICULUM)} prompts in parallel...")
all_prompts = asyncio.get_event_loop().run_until_complete(_collect_all())

dataset = Dataset.from_list(all_prompts)
print(f"Dataset: {len(dataset)} prompts")
print(dataset)

In [ ]:
# ── Cell 6: Reward function (multi-step, async-parallel) ──────────────────────

import math

def clamp(v: float, lo=0.01, hi=0.99) -> float:
    if not math.isfinite(v):
        return 0.1
    return max(lo, min(hi, v))


async def _eval_one_async(text: str, task: str, seed) -> float:
    """Run a full multi-step episode for one completion. Fully async."""
    actions = parse_all_actions(text)
    try:
        reward = await run_episode_async(task, actions, seed=seed)
        return clamp(reward)
    except Exception:
        return 0.1


def narada_reward(
    completions,
    prompts,
    task_type=None,
    seed=None,
    **kwargs,
):
    """Multi-step GRPO reward with async-parallel evaluation.

    Key upgrades over 1-step version:
    - parse_all_actions extracts the full 3-5 action plan from each completion
    - run_episode_async executes the entire plan; terminal reward is the signal
    - asyncio.gather runs all G=8 completions concurrently -- no sequential bottleneck

    Reward variance explanation:
    - Plan ends in correct flag:  ~0.95 terminal reward (strong positive)
    - Plan ends in wrong flag:    ~0.28 terminal reward (strong negative)
    - Plan ends in hops only:     ~0.47-0.56 step reward (weak, neutral-ish)
    - First visit to causal gene: +0.10 milestone added by environment
    reward_std across 8 completions is typically 0.15-0.35 -- much better than 1-step.

    TRL >= 0.9 passes completions as List[List[Dict]] (chat format).
    """
    n = len(completions)
    tasks = task_type if task_type is not None else ["monogenic"] * n
    seeds = seed if seed is not None else [None] * n

    texts = []
    for c in completions:
        if isinstance(c, list):
            texts.append(c[-1]["content"] if c else "")
        else:
            texts.append(str(c))

    async def _batch():
        return list(await asyncio.gather(*[
            _eval_one_async(t, task, s)
            for t, task, s in zip(texts, tasks, seeds)
        ]))

    return asyncio.get_event_loop().run_until_complete(_batch())


# Sanity check
test_plan = '{"action_type": "hop", "node_id": "HP:0000001", "reasoning": "Start from phenotype."}\n{"action_type": "summarise_trail", "reasoning": "Check path."}'
test_reward = narada_reward([test_plan], [[]], task_type=["monogenic"], seed=[1])
print(f"Sanity check reward: {test_reward[0]:.4f} (expect 0.45-0.55)")
print(f"Actions parsed from plan: {len(parse_all_actions(test_plan))}")

In [ ]:
# ── Cell 7: GRPO trainer setup ────────────────────────────────────────────────

from trl import GRPOConfig, GRPOTrainer
import shutil, os

# Clear Unsloth compiled cache — required when max_completion_length changes
cache_path = "/content/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cleared unsloth_compiled_cache")

grpo_config = GRPOConfig(
    # Core GRPO — G=8 + temp=1.1 forces diverse plans per prompt
    num_generations             = 8,
    temperature                 = 1.1,
    top_p                       = 0.95,

    # Optimiser
    learning_rate               = LR,
    per_device_train_batch_size = MINI_BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps                = WARMUP_STEPS,
    max_grad_norm               = 0.1,
    optim                       = "adamw_8bit",

    # Sequence — 800 tokens fits 4-5 JSON action blocks comfortably
    max_prompt_length           = 1200,
    max_completion_length       = 800,

    # Logging
    logging_steps               = 5,
    output_dir                  = f"outputs/{ADAPTER_NAME}",
    report_to                   = "none",
)

print("GRPOConfig ready. G=8, temp=1.1, max_completion_length=800")

In [ ]:
# -- Cell 8: Curriculum training --

from transformers import TrainerCallback, TrainerState, TrainerControl, TrainingArguments
import time

class RewardTracker(TrainerCallback):
    def __init__(self, phase, log):
        self.phase = phase
        self.log   = log
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        reward = logs.get("reward", logs.get("train/reward"))
        if reward is not None:
            self.log.append({"phase": self.phase, "step": state.global_step, "reward": float(reward)})

# -- Zero-shot baseline before any training --
print("="*60)
print("BASELINE (zero-shot, untrained LoRA weights)")
print("="*60)
FastLanguageModel.for_inference(model)
baseline_results = {}
for phase in CURRICULUM:
    task = phase["task"]
    scores = []
    for es in EVAL_SEEDS:
        steps = collect_episode(task, seed=es)
        if not steps: continue
        inputs = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user",   "content": steps[0]["prompt"]}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=200, temperature=0.3)
        completion = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        scores.append(run_episode(task, [parse_action(completion)], seed=es))
    avg = sum(scores) / len(scores) if scores else 0.0
    baseline_results[task] = avg
    print(f"Baseline {task}: {avg:.4f}  (n={len(scores)})")
FastLanguageModel.for_training(model)

# -- Curriculum training --
eval_results = {}
reward_log   = []
step_offset  = 0

for phase in CURRICULUM:
    task    = phase["task"]
    n_steps = phase["steps"]

    phase_data = dataset.filter(lambda x: x["task_type"] == task)
    if len(phase_data) == 0:
        print(f"Skipping {task} -- no data collected.")
        continue

    print(f"\n{'='*60}")
    print(f"Phase: {task}  |  {len(phase_data)} prompts  |  {n_steps} steps")
    print(f"{'='*60}")

    grpo_config.max_steps = n_steps
    tracker = RewardTracker(task, reward_log)

    trainer = GRPOTrainer(
        model            = model,
        processing_class = tokenizer,
        reward_funcs     = narada_reward,
        args             = grpo_config,
        train_dataset    = phase_data,
        callbacks        = [tracker],
    )

    t0 = time.time()
    trainer.train()
    step_offset += n_steps
    print(f"Phase {task} done in {(time.time()-t0)/60:.1f} min")

    FastLanguageModel.for_inference(model)
    scores = []
    for es in EVAL_SEEDS:
        steps = collect_episode(task, seed=es)
        if not steps: continue
        inputs = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user",   "content": steps[0]["prompt"]}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=200, temperature=0.3)
        completion = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        scores.append(run_episode(task, [parse_action(completion)], seed=es))
    avg = sum(scores) / len(scores) if scores else 0.0
    eval_results[task] = avg
    print(f"Eval {task}: {avg:.4f}  (n={len(scores)})")
    FastLanguageModel.for_training(model)

print("\n=== CURRICULUM COMPLETE ===")
print(f"{'Task':<25} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-"*57)
for task in [p["task"] for p in CURRICULUM]:
    b = baseline_results.get(task, 0.0)
    t = eval_results.get(task, 0.0)
    print(f"  {task:<23} {b:>10.4f} {t:>10.4f} {t-b:>+10.4f}")

In [ ]:
# ── Cell 9: Save adapter ──────────────────────────────────────────────────────

model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Adapter saved to ./{ADAPTER_NAME}")

# Push to HF Hub (optional — set HF_PUSH_REPO above)
if HF_PUSH_REPO:
    model.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
    print(f"Adapter pushed to https://huggingface.co/{HF_PUSH_REPO}")
else:
    print("Set HF_PUSH_REPO to push the adapter to HF Hub.")

In [ ]:
# -- Cell 9.5: Training curves + before/after bar chart --
import matplotlib.pyplot as plt

colors = {"monogenic": "#2196F3", "oligogenic": "#FF9800", "phenotype_mismatch": "#E91E63"}

# Fig 1: reward over all steps
fig, ax = plt.subplots(figsize=(10, 5))
global_step = 0
for phase in CURRICULUM:
    task = phase["task"]
    pts  = [(e["step"] + global_step, e["reward"]) for e in reward_log if e["phase"] == task]
    if pts:
        xs, ys = zip(*pts)
        ax.plot(xs, ys, "o-", color=colors[task], label=task, linewidth=1.5, markersize=3)
    global_step += phase["steps"]
ax.set_xlabel("Training step")
ax.set_ylabel("Mean reward")
ax.set_title("Narada GRPO -- reward curve (curriculum order)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("training_curve.png", dpi=150); plt.show()

# Fig 2: before/after bar chart
tasks_list = [p["task"] for p in CURRICULUM]
x = range(len(tasks_list)); w = 0.35
fig2, ax2 = plt.subplots(figsize=(9, 5))
ax2.bar([i - w/2 for i in x], [baseline_results.get(t, 0) for t in tasks_list], w,
        label="Zero-shot baseline", color="#90A4AE")
ax2.bar([i + w/2 for i in x], [eval_results.get(t, 0) for t in tasks_list], w,
        label="After GRPO", color="#43A047")
ax2.set_xticks(list(x)); ax2.set_xticklabels(tasks_list)
ax2.set_ylabel("Avg reward (5 eval seeds)")
ax2.set_title("Narada -- zero-shot vs GRPO-trained (Qwen3-1.7B)")
ax2.set_ylim(0, 1.0); ax2.legend(); ax2.grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("before_after.png", dpi=150); plt.show()

print("Plots saved: training_curve.png, before_after.png")
print("Download from the Colab file browser and commit to the repo.")

In [ ]:
# ── Cell 10: Full-episode evaluation (post-training) ─────────────────────────
# Run 3 full episodes (one per task) with the trained model.
# Each episode runs until done or max_steps.

from IPython.display import display
import pandas as pd

FastLanguageModel.for_inference(model)

async def run_full_episode_async(task_type: str, seed: int = 0) -> Dict[str, Any]:
    ws_url = ENV_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"
    history = []
    conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

    async with websockets.connect(ws_url, open_timeout=30, ping_interval=20) as ws:
        await ws.send(json.dumps({"type": "reset", "task_type": task_type, "seed": seed}))
        raw = json.loads(await ws.recv())
        obs = raw["data"]["observation"]

        while not obs["done"] and obs["step"] < obs["max_steps"]:
            user_text = format_obs(obs)
            conversation.append({"role": "user", "content": user_text})

            inputs = tokenizer.apply_chat_template(
                conversation, tokenize=True, add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")
            with torch.no_grad():
                out = model.generate(inputs, max_new_tokens=250, temperature=0.2)
            completion = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
            conversation.append({"role": "assistant", "content": completion})

            action = parse_action(completion)
            await ws.send(json.dumps({"type": "step", "action": action}))
            raw = json.loads(await ws.recv())
            data = raw["data"]
            obs = data["observation"]
            history.append({"step": obs["step"], "action": action["action_type"], "reward": data["reward"]})

    return {"task": task_type, "steps": len(history), "final_reward": history[-1]["reward"] if history else 0.0, "history": history}


results = []
for task in ["monogenic", "oligogenic", "phenotype_mismatch"]:
    r = asyncio.get_event_loop().run_until_complete(run_full_episode_async(task, seed=42))
    results.append(r)
    print(f"{task:25s}: {r['steps']} steps, final_reward={r['final_reward']:.4f}")

display(pd.DataFrame([{"task": r["task"], "steps": r["steps"], "final_reward": r["final_reward"]} for r in results]))

In [ ]:
# ── Cell 11: Summary ──────────────────────────────────────────────────────────

print("╔══════════════════════════════════════════════╗")
print("║         NARADA TRAINING COMPLETE              ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Base model : {BASE_MODEL:<29} ║")
print(f"║  Adapter    : {ADAPTER_NAME:<29} ║")
print("╠══════════════════════════════════════════════╣")
print("║  Post-training eval scores                    ║")
for task, score in eval_results.items():
    print(f"║    {task:25s}: {score:.4f}          ║")
print("╚══════════════════════════════════════════════╝")

if HF_PUSH_REPO:
    print(f"\nModel available at: https://huggingface.co/{HF_PUSH_REPO}")